In [1]:
import pandas as pd
import os

In [2]:
def Read_data(path):
    df_2004 = pd.read_csv(f'{path}\\data\\EI_2004_2013(Sustancias).csv')
    df_2014 = pd.read_csv(f'{path}\\data\\ECERIECSLegalesIlegales_mod(EdadInicio).csv')
    dfconcat = pd.concat([df_2004, df_2014], ignore_index=True)
    return dfconcat

def safe_convert_to_float(value):
    try:
        return float(value)
    except ValueError:
        return None

def Grupos(df):
    cols_derecha_EdadInicio = [
    # Marihuana
    'EdadInicioMarihuana', 'EdadInicioHachis',

    # Cocaína
    'EdadInicioCocaína', 'EdadInicioCrack', 'EdadInicioOtrasPresentaciones_Basuco_o_pasta_base_cocaina_negra',

    # Inhalables
    'EdadInicioSolventes_y_removedores', 'EdadInicioPegamento',
    'EdadInicioEsmaltes_y_pinturas', 'EdadInicioOtros_aire_comprimido_gasolinas_combustibles',

    # Metanfetaminas
    'EdadInicioAnfetaminas', 'EdadInicioMetanfetaminas',
    'EdadInicioMDMA_extasis_y_metanfetaminas_alucinogenas_DMT',
    'EdadInicioOtros_derivados_anfetaminicos',

    # Alucinógenos
    'EdadInicioLSD', 'EdadInicioPlantas_alucinogenas_y_derivados',
    'EdadInicioOtras_PCP_ketamina_excep_metanfetamina',

    # Medicamentos
    'EdadInicioBenzodiazepinas', 'EdadInicioRohypnol',
    'EdadInicioOtras_sedantes_hipnoticos_GHB',
    'EdadInicioCon_utilidad_medica_Prozac_Paxil_Carbamazepina',

    # Opioides
    'EdadInicioHeroina', 'EdadInicioOpiaceos_sinteticos_propoxifeno_nailbufina',
    'EdadInicioOpio_y_opiodes_morfina_codeina'
]

    for col in cols_derecha_EdadInicio:
        df[col] = df[col].apply(safe_convert_to_float)
    # Agrupación por edad de inicio
    df['EdadInicioMarihuana'] = df[['EdadInicioMarihuana', 'EdadInicioHachis']].min(axis=1)
    df['EdadInicioCocaína'] = df[['EdadInicioCocaína', 'EdadInicioCrack', 'EdadInicioOtrasPresentaciones_Basuco_o_pasta_base_cocaina_negra']].min(axis=1)
    df['EdadInicioInhalables'] = df[['EdadInicioSolventes_y_removedores', 'EdadInicioPegamento', 'EdadInicioEsmaltes_y_pinturas', 'EdadInicioOtros_aire_comprimido_gasolinas_combustibles']].min(axis=1)
    df['EdadInicioMetanfetaminas'] = df[['EdadInicioAnfetaminas', 'EdadInicioMetanfetaminas', 'EdadInicioMDMA_extasis_y_metanfetaminas_alucinogenas_DMT', 'EdadInicioOtros_derivados_anfetaminicos']].min(axis=1)
    df['EdadInicioAlucinógenos'] = df[['EdadInicioLSD', 'EdadInicioPlantas_alucinogenas_y_derivados', 'EdadInicioOtras_PCP_ketamina_excep_metanfetamina']].min(axis=1)
    df['EdadInicioMedicamentos'] = df[['EdadInicioBenzodiazepinas', 'EdadInicioRohypnol', 'EdadInicioOtras_sedantes_hipnoticos_GHB', 'EdadInicioCon_utilidad_medica_Prozac_Paxil_Carbamazepina']].min(axis=1)
    df['EdadInicioOpioides'] = df[['EdadInicioHeroina', 'EdadInicioOpiaceos_sinteticos_propoxifeno_nailbufina', 'EdadInicioOpio_y_opiodes_morfina_codeina']].min(axis=1)

    # Columnas a eliminar (las que ya se agruparon)
    cols_edad_no_finales = [
        'EdadInicioHachis',
        'EdadInicioCrack',
        'EdadInicioOtrasPresentaciones_Basuco_o_pasta_base_cocaina_negra',
        'EdadInicioSolventes_y_removedores',
        'EdadInicioPegamento',
        'EdadInicioEsmaltes_y_pinturas',
        'EdadInicioOtros_aire_comprimido_gasolinas_combustibles',
        'EdadInicioAnfetaminas',
        'EdadInicioMDMA_extasis_y_metanfetaminas_alucinogenas_DMT',
        'EdadInicioOtros_derivados_anfetaminicos',
        'EdadInicioLSD',
        'EdadInicioPlantas_alucinogenas_y_derivados',
        'EdadInicioOtras_PCP_ketamina_excep_metanfetamina',
        'EdadInicioBenzodiazepinas',
        'EdadInicioRohypnol',
        'EdadInicioOtras_sedantes_hipnoticos_GHB',
        'EdadInicioCon_utilidad_medica_Prozac_Paxil_Carbamazepina',
        'EdadInicioHeroina',
        'EdadInicioOpiaceos_sinteticos_propoxifeno_nailbufina',
        'EdadInicioOpio_y_opiodes_morfina_codeina'
    ]

    df.drop(columns=cols_edad_no_finales, inplace=True)
    return df

def generar_orden_consumo(df):
    # columnas de edad de inicio
    edad_cols = [col for col in df.columns if col.startswith('EdadInicio')]

    # para cada fila, ordenamos las edades válidas (mayores a 0, no NaN)
    for idx, row in df.iterrows():
        # filtrar valores válidos
        edades_validas = {col: row[col] for col in edad_cols if pd.notna(row[col]) and row[col] > 0}

        # ordenar por edad
        ordenadas = sorted(edades_validas.items(), key=lambda x: x[1])

        # asignar orden (1,2,3...)
        for orden, (col, _) in enumerate(ordenadas, start=1):
            col_orden = col.replace('EdadInicio', 'OrdenConsumo')
            df.loc[idx, col_orden] = orden

    # rellenar las columnas de orden que faltan con 0
    orden_cols = [col.replace('EdadInicio', 'OrdenConsumo') for col in edad_cols]
    for col in orden_cols:
        if col not in df.columns:
            df[col] = 0

    return df

# por grupo de sustancia
def construir_orden(df, grupos):
    columnas_orden = [f"OrdenConsumo{g}" for g in grupos]
    
    transiciones = []

    for idx, fila in df[columnas_orden].iterrows():
        orden = fila.dropna().astype(int).to_dict()
        orden_invertido = {v: k for k, v in orden.items()}

        if 1 in orden_invertido and 2 in orden_invertido:
            sustancia_1 = f"{orden_invertido[1]}1"
            sustancia_2 = f"{orden_invertido[2]}2"
            transiciones.append((sustancia_1, sustancia_2))

        if 2 in orden_invertido and 3 in orden_invertido:
            sustancia_2 = f"{orden_invertido[2]}2"
            sustancia_3 = f"{orden_invertido[3]}3"
            transiciones.append((sustancia_2, sustancia_3))

        if 3 in orden_invertido and 4 in orden_invertido:
            sustancia_3 = f"{orden_invertido[3]}3"
            sustancia_4 = f"{orden_invertido[4]}4"
            transiciones.append((sustancia_3, sustancia_4))
            
        if 4 in orden_invertido and 5 in orden_invertido:
            sustancia_4 = f"{orden_invertido[4]}4"
            sustancia_5 = f"{orden_invertido[5]}5"
            transiciones.append((sustancia_4, sustancia_5))


    df_orden = pd.DataFrame(transiciones, columns=["from", "to"])
    tabla = df_orden.value_counts().reset_index(name="value")

    return tabla

In [3]:
def main():
    path = os.getcwd()
    dfconcat = Read_data(path)

    dfconcat = Grupos(dfconcat)
    listEdadInicio = [col for col in dfconcat.columns if 'EdadInicio' in col]
    for col in listEdadInicio:
        dfconcat[col] = dfconcat[col].apply(safe_convert_to_float)
    dfconcat = dfconcat[['FolioId'] + listEdadInicio]
    dfconcat = generar_orden_consumo(dfconcat)
    sustancias = ['Alcohol', 'Otras Sustancias', 'Tabaco', 'Marihuana',
            'Cocaína', 'Inhalables', 'Metanfetaminas',
            'Alucinógenos', 'Medicamentos', 'Opioides']
    cols_edad = [f"EdadInicio{s}" for s in sustancias]
    cols_orden = [f"OrdenConsumo{s}" for s in sustancias]
    dfconcatfinal = dfconcat[['FolioId'] + cols_edad + cols_orden].copy()
    dfconcatfinal = construir_orden(dfconcatfinal, sustancias)
    return dfconcatfinal

In [4]:
if __name__ == "__main__":
    dfconcat = main()

C:\Users\franc\AppData\Local\Temp\ipykernel_18540\2735233801.py:2: DtypeWarning: Columns (95,98,99,101,102,103,104,106,107,109,110,111,112,113,118,119) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2004 = pd.read_csv(f'{path}\\data\\EI_2004_2013(Sustancias).csv')


In [5]:
dfconcat.insert(0,'Causa','causita')
dfconcat['from'] = dfconcat['from'].str.replace('OrdenConsumo', '', regex=False)
dfconcat['to'] = dfconcat['to'].str.replace('OrdenConsumo', '', regex=False)
dfconcat.to_csv('Sankey_Historico.csv', index=False)

In [6]:
dfconcat

,Causa,from,to,value
0,causita,Tabaco1,Alcohol2,156374
1,causita,Alcohol1,Tabaco2,77187
2,causita,Alcohol2,Marihuana3,74403
3,causita,Tabaco2,Marihuana3,37986
4,causita,Tabaco1,Marihuana2,27455
...,...,...,...,...
346,causita,Otras Sustancias1,Metanfetaminas2,1
347,causita,Otras Sustancias2,Cocaína3,1
348,causita,Otras Sustancias2,Opioides3,1
349,causita,Otras Sustancias1,Alucinógenos2,1
